In [163]:
# LINKING_V4 — L1: load raw ROI paths + imaging sheet

from pathlib import Path
import pandas as pd

ROOT = Path("/Users/davekokel/Projects/carp_v2")
if not ROOT.exists():
    raise FileNotFoundError(f"Expected repo root at {ROOT}, but it does not exist.")

BASE    = ROOT / "seed_kits" / "legacy_wrangling_v2"
RAW     = BASE / "raw"
WORKING = BASE / "working"

for p in [RAW, WORKING]:
    if not p.exists():
        raise FileNotFoundError(f"Expected directory at {p}, but it does not exist.")

ROI_PATHS_FILE   = RAW / "2025-11-13-092338-korra_aang_roi_root_tiffs_good-3.xlsx"
IMAGING_SHEET_FILE = RAW / "2025-11-21-220012-imaging_sheet.xlsx"

print("ROI paths file:     ", ROI_PATHS_FILE)
print("Imaging sheet file: ", IMAGING_SHEET_FILE)

df_roi_paths = pd.read_excel(ROI_PATHS_FILE)
df_imaging   = pd.read_excel(IMAGING_SHEET_FILE)

print("df_roi_paths shape:", df_roi_paths.shape)
print("df_imaging shape: ", df_imaging.shape)
print("df_roi_paths columns:", list(df_roi_paths.columns))
print("df_imaging columns:", list(df_imaging.columns))


ROI paths file:      /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/2025-11-13-092338-korra_aang_roi_root_tiffs_good-3.xlsx
Imaging sheet file:  /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/2025-11-21-220012-imaging_sheet.xlsx
df_roi_paths shape: (976, 7)
df_imaging shape:  (347, 28)
df_roi_paths columns: ['date_experiment', 'fish', 'roi_rel', 'roi_name', 'roi_tiffs', 'roi_dir', 'dataset']
df_imaging columns: ['date_mount', 'mount_id', 'ZF female genotype', 'ZF male genotype', 'additional plasmids injected', 'additional mRNAs injected', 'additonal proteins injected', 'additonal dye and chemicals', 'Date born', 'Time mounted', 'Mounting Orientation', 'Date screened/Initial feedback', 'Date imaged', 'Time placed in scope', 'Start of imaging time', 'End of imaging time', 'Imaged Locations', 'Unique Targets with blanks', 'Unique Targets', 'Data location', 'Dataset size (GB) - raw data only', 'Camera Filters', 'JSON excite map for ZF male', 'JSON 

In [164]:
# L2 — Normalize slugs and dates on ROI side and imaging-sheet side

import re
import pandas as pd

print("L2 — starting slug/date normalization")

# ─────────────────────────────────────────
# Helper: canonical slug normalizer
# ─────────────────────────────────────────
def normalize_slug_raw(s: str | None) -> str | None:
    """
    Normalize experiment slugs so that:
      - 20250513_skittles == 20250513_skittles(fish1-3) == 20250513_skittles_mount1
      - mem-histone == mem_histone
    """
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    s = str(s).strip().lower()

    # strip after first space
    s = s.split(" ")[0]

    # strip parenthetical suffixes: (fish1-3), (mount1), etc.
    s = re.sub(r"\(.*\)$", "", s)

    # strip explicit mount suffixes: _mount1, _mount2
    s = re.sub(r"_mount\d+$", "", s)

    # unify hyphen/underscore
    s = s.replace("-", "_")

    # collapse multiple underscores
    s = re.sub(r"__+", "_", s)

    # trim trailing underscores
    s = s.strip("_")

    return s or None

# ─────────────────────────────────────────
# ROI side: dataset_slug_norm + roi_path_date_yyyymmdd
# ─────────────────────────────────────────

# ROI paths file has columns:
# ['date_experiment', 'fish', 'roi_rel', 'roi_name', 'roi_tiffs', 'roi_dir', 'dataset']

# 1) experiment slug from ROI (we reuse the 'dataset' column, e.g. '20250428_mem_histone')
df_roi_paths["dataset_slug_raw"] = df_roi_paths["dataset"]

# 2) normalized slug
df_roi_paths["dataset_slug_norm"] = df_roi_paths["dataset_slug_raw"].apply(normalize_slug_raw)

# 3) date from roi_dir (YYYYMMDD prefix in folder name)
def roi_date_from_path(s: str | None) -> str | None:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    s = str(s)
    m = re.search(r"/(\d{8})_", s)
    if m:
        return m.group(1)
    return None

df_roi_paths["roi_path_date_yyyymmdd"] = df_roi_paths["roi_dir"].apply(roi_date_from_path)

print("\nL2 — ROI side sample:")
print(
    df_roi_paths[
        ["roi_dir", "dataset_slug_raw", "dataset_slug_norm", "roi_path_date_yyyymmdd"]
    ].head(10)
)

# ─────────────────────────────────────────
# Imaging side: sheet_slug_norm + date_mount_yyyymmdd
# ─────────────────────────────────────────

# 1) get slug from Data location:
#    X:\abcabc\Korra_Foundation\20250513_skittles(fish1-3)
def slug_from_data_location(s: str | None) -> str | None:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    s = str(s)
    m = re.search(r"\\(Korra_Foundation|Aang_Foundation)\\([^\\]+)", s)
    if m:
        return m.group(2)
    return None

df_imaging["sheet_slug_raw"] = df_imaging["Data location"].apply(slug_from_data_location)
df_imaging["sheet_slug_norm"] = df_imaging["sheet_slug_raw"].apply(normalize_slug_raw)

# 2) choose the mount-date column from the imaging sheet
date_mount_col = None
for cand in ["date_mount", "Date mount", "Date mounted", "Date mounted?"]:
    if cand in df_imaging.columns:
        date_mount_col = cand
        break

if date_mount_col is None:
    raise KeyError(
        "Could not find a date_mount-like column in imaging sheet; "
        "expected one of ['date_mount', 'Date mount', 'Date mounted', 'Date mounted?']"
    )

df_imaging["date_mount_yyyymmdd"] = pd.to_datetime(
    df_imaging[date_mount_col], errors="coerce"
).dt.strftime("%Y%m%d")

print("\nL2 — imaging side sample:")
print(
    df_imaging[
        ["Data location", "sheet_slug_raw", "sheet_slug_norm", date_mount_col, "date_mount_yyyymmdd"]
    ].head(10)
)

L2 — starting slug/date normalization

L2 — ROI side sample:
                                             roi_dir dataset_slug_raw  \
0  /clusterfs/vast/abcabc/Korra_Foundation/202504...            korra   
1  /clusterfs/vast/abcabc/Korra_Foundation/202504...            korra   
2  /clusterfs/vast/abcabc/Korra_Foundation/202504...            korra   
3  /clusterfs/vast/abcabc/Korra_Foundation/202504...            korra   
4  /clusterfs/vast/abcabc/Korra_Foundation/202504...            korra   
5  /clusterfs/vast/abcabc/Korra_Foundation/202504...            korra   
6  /clusterfs/vast/abcabc/Korra_Foundation/202504...            korra   
7  /clusterfs/vast/abcabc/Korra_Foundation/202504...            korra   
8  /clusterfs/vast/abcabc/Korra_Foundation/202504...            korra   
9  /clusterfs/vast/abcabc/Korra_Foundation/202504...            korra   

  dataset_slug_norm roi_path_date_yyyymmdd  
0             korra               20250428  
1             korra               20250428  


In [165]:
# L-debug — QC manual “hole” ROIs vs imaging sheet (slug + date_mount)

from pathlib import Path
import pandas as pd
import re

ROOT = Path("/Users/davekokel/Projects/carp_v2")
RAW_V2 = ROOT / "seed_kits" / "legacy_wrangling_v2" / "raw"

MANUAL_PATH    = RAW_V2 / "roi_missing_parents_for_manual_mapping_DQM_CNH (1).csv"
IMAGING_PATH   = RAW_V2 / "2025-11-21-220012-imaging_sheet.xlsx"

print("MANUAL:", MANUAL_PATH)
print("IMAGING:", IMAGING_PATH)

manual = pd.read_csv(MANUAL_PATH)
sheet  = pd.read_excel(IMAGING_PATH)

print("manual shape:", manual.shape)
print("sheet shape:", sheet.shape)

# --- 1) derive slug + date from manual ROI side ---

def slug_from_roi_dir(s: str) -> str:
    if pd.isna(s):
        return None
    s = str(s)
    m = re.search(r"/(\d{8}_[^/]+)/", s)
    if m:
        return m.group(1)
    # fallback: use folder name under Foundation
    m2 = re.search(r"/(Aang_Foundation|Korra_Foundation)/([^/]+)/", s)
    if m2:
        return m2.group(2)
    return None

def date_from_roi_dir(s: str) -> str:
    if pd.isna(s):
        return None
    s = str(s)
    m = re.search(r"/(\d{8})_", s)
    if m:
        return m.group(1)
    return None

manual["roi_slug"]     = manual["roi_dir"].apply(slug_from_roi_dir)
manual["roi_date_str"] = manual["roi_dir"].apply(date_from_roi_dir)

print("\nL-debug — manual unique slugs/dates (first 20):")
display(
    manual[["roi_slug", "roi_date_str"]]
    .drop_duplicates()
    .head(20)
)

# --- 2) derive slug + date_mount from imaging sheet ---

def slug_from_data_location(s: str) -> str:
    if pd.isna(s):
        return None
    s = str(s)
    # imaging sheet paths look like X:\abcabc\Korra_Foundation\20250513_skittles(fish1-3)
    m = re.search(r"\\(Korra_Foundation|Aang_Foundation)\\([^\\]+)", s)
    if m:
        return m.group(2)
    return None

sheet["sheet_slug"] = sheet["Data location"].apply(slug_from_data_location)

# pick whichever column in the sheet is the true mount date
date_mount_col = None
for cand in ["date_mount", "Date mount", "Date mounted", "Date mounted?"]:
    if cand in sheet.columns:
        date_mount_col = cand
        break

if date_mount_col is None:
    raise KeyError("Could not find a date_mount-like column in imaging sheet.")

sheet["date_mount_yyyymmdd"] = pd.to_datetime(
    sheet[date_mount_col], errors="coerce"
).dt.strftime("%Y%m%d")

print("\nL-debug — imaging unique (sheet_slug, date_mount) combos (first 20):")
display(
    sheet[["sheet_slug", "date_mount_yyyymmdd"]]
    .drop_duplicates()
    .head(20)
)

# --- 3) join manual slugs to sheet slugs on slug + date_mount ---

def norm_slug(s):
    if pd.isna(s):
        return None
    return str(s).strip().lower()

manual["roi_slug_norm"]  = manual["slug"].fillna(manual["roi_slug"]).apply(norm_slug)
sheet["sheet_slug_norm"] = sheet["sheet_slug"].apply(norm_slug)

hole_slugs = sorted(manual["roi_slug_norm"].dropna().unique())
print("\nL-debug — hole slugs from manual sheet:")
print(hole_slugs)

summary_rows = []
for slug in hole_slugs:
    sub_sheet = sheet[sheet["sheet_slug_norm"] == slug]
    if sub_sheet.empty:
        summary_rows.append((slug, 0, None))
        continue

    counts = (
        sub_sheet["date_mount_yyyymmdd"]
        .value_counts(dropna=False)
        .reset_index()
        .rename(columns={"index": "date_mount_yyyymmdd", "date_mount_yyyymmdd": "n_rows"})
    )
    summary_rows.append((slug, len(sub_sheet), counts))

# display summary
print("\nL-debug — imaging-sheet matches per slug:")
for slug, n_rows, counts_df in summary_rows:
    print(f"  slug={slug!r} → {n_rows} imaging rows")
    if counts_df is not None:
        print(counts_df.head(10))

# --- 4) show detailed matches for any slug with >1 date_mount or >1 row per date_mount ---

print("\nL-debug — detailed imaging rows for ambiguous slugs (multiple mounts per slug):")
for slug, n_rows, counts_df in summary_rows:
    if counts_df is None:
        continue
    # slug with > 1 distinct date_mount or any date_mount with >1 row
    bad = counts_df[counts_df["n_rows"] > 1]
    if len(counts_df) > 1 or not bad.empty:
        sub_sheet = sheet[sheet["sheet_slug_norm"] == slug]
        print(f"\n=== slug {slug!r} ===")
        display(
            sub_sheet[
                [
                    "sheet_slug",
                    date_mount_col,
                    "date_mount_yyyymmdd",
                    "Data location",
                ]
            ].head(50)
        )

MANUAL: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/roi_missing_parents_for_manual_mapping_DQM_CNH (1).csv
IMAGING: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/2025-11-21-220012-imaging_sheet.xlsx
manual shape: (172, 6)
sheet shape: (347, 28)

L-debug — manual unique slugs/dates (first 20):


,roi_slug,roi_date_str
0,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,20250721
5,20250723_er-mSG_mem-mChilada,20250723
6,20251002_mem_mito,20251002
30,20251003_mem_tester,20251003
37,20251006_mem_histone,20251006
48,20251007_mem_histone,20251007
55,20251009_mem-mito,20251009
59,20251107_mem-kinectocore,20251107
60,20251028_peroxi,20251028
62,20251029_mem-histone,20251029



L-debug — imaging unique (sheet_slug, date_mount) combos (first 20):


,sheet_slug,date_mount_yyyymmdd
0,None,20251121
1,20251121_mem-histone,20251121
2,20251117_mem-histone,20251117
3,20251113_mem-mito_mount1,20251114
4,20251113_mem-mito_mount2,20251114
5,20251113_mem-histone,20251113
6,20251112_mem-mito,20251112
7,None,20251112
8,20251107_mem-mito,20251111
9,20251110_mem-histone,20251110



L-debug — hole slugs from manual sheet:
['20250513_skittles', '20250522_skittlez', '20250528_skittlez', '20250625_skittlez', '20250627_skittlez', '20250721_72hpf_mrna_msg_organelle_lls-sim', '20250724_mem-mchilada_er-msg', '20251002_mem_mito', '20251003_mem_tester', '20251006_mem_histone', '20251007_mem-histone', '20251007_mem_histone', '20251009_mem-mito', '20251028_peroxi', '20251029_mem-histone', '20251029_peroxi', 'analysis_test', 'er-msg_mem-mchilada', 'mem-kinetocore', 'mem-mchilada_er-msg', 'mem_histone', 'mem_mito', 'mem_tester', 'mitomsg', 'mrna_msg_organelle_lls-sim/er_roi1', 'peroxi', 'skittes', 'skittlez']

L-debug — imaging-sheet matches per slug:
  slug='20250513_skittles' → 0 imaging rows
  slug='20250522_skittlez' → 1 imaging rows
     n_rows  count
0  20250522      1
  slug='20250528_skittlez' → 1 imaging rows
     n_rows  count
0  20250528      1
  slug='20250625_skittlez' → 0 imaging rows
  slug='20250627_skittlez' → 0 imaging rows
  slug='20250721_72hpf_mrna_msg_or

TypeError: '>' not supported between instances of 'str' and 'int'

In [166]:
# LINKING_V4 — L2a: extract roi_path_date_yyyymmdd from roi_dir

import re
import math

def extract_roi_path_date(path):
    """
    Extract YYYYMMDD from ROI path segments like:
      .../20250428_mem_histone/fish1_72hpf/roi1_tail
      .../20250627_skittlez/fishX/roiY

    Rule:
      Find a path component that matches ^\d{8}_.+ and return the 8-digit prefix.
    """
    if path is None or (isinstance(path, float) and pd.isna(path)):
        return None

    s = str(path)
    parts = s.split("/")
    for p in parts:
        m = re.match(r"^(\d{8})_.+", p)
        if m:
            return m.group(1)
    return None

df_roi_paths["roi_path_date_yyyymmdd"] = df_roi_paths["roi_dir"].apply(extract_roi_path_date)

n_total   = len(df_roi_paths)
n_nonnull = df_roi_paths["roi_path_date_yyyymmdd"].notna().sum()
n_null    = n_total - n_nonnull
n_unique  = df_roi_paths["roi_path_date_yyyymmdd"].nunique()

print("L2a — ROI path date extraction QC")
print("  total rows:           ", n_total)
print("  non-null dates:       ", n_nonnull)
print("  null dates:           ", n_null)
print("  unique dates:         ", n_unique)

if n_null > 0:
    print("\nL2a: sample rows with NULL roi_path_date_yyyymmdd:")
    print(df_roi_paths[df_roi_paths["roi_path_date_yyyymmdd"].isna()][["roi_dir"]].head(20))

print("\nL2a: sample ROI rows with parsed dates:")
print(df_roi_paths[["roi_dir", "roi_path_date_yyyymmdd"]].head(20))

L2a — ROI path date extraction QC
  total rows:            976
  non-null dates:        975
  null dates:            1
  unique dates:          76

L2a: sample rows with NULL roi_path_date_yyyymmdd:
                                               roi_dir
419  /clusterfs/vast/abcabc/Korra_Foundation/analys...

L2a: sample ROI rows with parsed dates:
                                              roi_dir roi_path_date_yyyymmdd
0   /clusterfs/vast/abcabc/Korra_Foundation/202504...               20250428
1   /clusterfs/vast/abcabc/Korra_Foundation/202504...               20250428
2   /clusterfs/vast/abcabc/Korra_Foundation/202504...               20250429
3   /clusterfs/vast/abcabc/Korra_Foundation/202504...               20250429
4   /clusterfs/vast/abcabc/Korra_Foundation/202504...               20250429
5   /clusterfs/vast/abcabc/Korra_Foundation/202504...               20250429
6   /clusterfs/vast/abcabc/Korra_Foundation/202504...               20250429
7   /clusterfs/vast/abcabc/Korra_F

<>:13: SyntaxWarning: invalid escape sequence '\d'
<>:13: SyntaxWarning: invalid escape sequence '\d'
/var/folders/29/cdrb2nrn01s4d1_n6gvxy5j80000gn/T/ipykernel_45674/1009012992.py:13: SyntaxWarning: invalid escape sequence '\d'
  Find a path component that matches ^\d{8}_.+ and return the 8-digit prefix.


In [167]:
# LINKING_V4 — L2b: drop ROI rows that do not contain a parsable experiment date

mask_valid = df_roi_paths["roi_path_date_yyyymmdd"].notna()

n_total = len(df_roi_paths)
n_keep  = mask_valid.sum()
n_drop  = n_total - n_keep

print("L2b — filter non-experiment ROIs")
print("  total ROI rows:", n_total)
print("  keeping       :", n_keep)
print("  dropping      :", n_drop)

if n_drop > 0:
    print("\nL2b: sample of dropped roi_dir values:")
    print(df_roi_paths.loc[~mask_valid, ["roi_dir"]].head(20))

df_roi_paths = df_roi_paths.loc[mask_valid].reset_index(drop=True)

print("\nL2b: df_roi_paths shape after filter:", df_roi_paths.shape)

L2b — filter non-experiment ROIs
  total ROI rows: 976
  keeping       : 975
  dropping      : 1

L2b: sample of dropped roi_dir values:
                                               roi_dir
419  /clusterfs/vast/abcabc/Korra_Foundation/analys...

L2b: df_roi_paths shape after filter: (975, 10)


In [168]:
# LINKING_V4 — L2c: parse roi_dir into experiment_folder, fish_folder, fish_id, fish_number,
#                    fish_age, roi_index, and fish_nickname using / and _ tokenization.

import re
import pandas as pd

df_roi_paths = df_roi_paths.copy()

# --- basic path-level pieces -------------------------------------------------

def extract_experiment_folder(path: str) -> str:
    """
    From roi_dir like:
      .../20250428_mem_histone/fish1_72hpf/roi1_tail
      .../20250606_skittlez/fish10_roi1
    pick the experiment folder:
      -> 20250428_mem_histone
      -> 20250606_skittlez
    """
    parts = str(path).split("/")
    for p in parts:
        if re.match(r"\d{8}_.+", p):
            return p
    return None

def extract_fish_folder(path: str) -> str:
    """
    FISH folder is the directory immediately under experiment_folder.
    For example:
      .../20250428_mem_histone/fish1_72hpf/roi1_tail -> 'fish1_72hpf'
      .../20250606_skittlez/fish10_roi1               -> 'fish10_roi1'
    """
    parts = str(path).split("/")
    for i, p in enumerate(parts):
        if re.match(r"\d{8}_.+", p) and i + 1 < len(parts):
            return parts[i + 1]
    return None

def extract_roi_folder(path: str) -> str:
    """
    ROI folder is the last path segment.
      .../fish1_24hpf/roi1_tail -> 'roi1_tail'
      .../20250606_skittlez/fish10_roi1 -> 'fish10_roi1'
    """
    return str(path).rstrip("/").split("/")[-1]

df_roi_paths["experiment_folder"] = df_roi_paths["roi_dir"].apply(extract_experiment_folder)
df_roi_paths["fish_folder"]       = df_roi_paths["roi_dir"].apply(extract_fish_folder)
df_roi_paths["roi_folder"]        = df_roi_paths["roi_dir"].apply(extract_roi_folder)

# --- tokenization + inference helpers ----------------------------------------

def tokenize_path(path: str):
    """
    Split roi_dir into lowercase tokens using both '/' and '_' as separators.
    Returns a list of tokens.
    """
    s = str(path).replace("\\", "/").lower()
    # split on / first, then split each segment on _
    segments = s.split("/")
    tokens = []
    for seg in segments:
        tokens.extend(seg.split("_"))
    # filter out empty strings
    return [t for t in tokens if t]

def infer_fish_number(tokens):
    """
    Look for fish number in tokens.
    Patterns:
      - 'fish7'          -> 7
      - ['fish', '7']    -> 7
    Returns int or None.
    """
    # fishXX
    for t in tokens:
        m = re.match(r"^fish\s*0*([0-9]+)$", t)
        if m:
            return int(m.group(1))

    # 'fish', '7'
    for i, t in enumerate(tokens[:-1]):
        if t == "fish":
            m = re.match(r"^0*([0-9]+)$", tokens[i+1])
            if m:
                return int(m.group(1))

    return None

def infer_fish_age(tokens):
    """
    Look for age tokens like '72hpf', '24hpf', etc.
    Returns (age_token, age_hpf:int) or (None, None).
    """
    for t in tokens:
        m = re.match(r"^([0-9]+)hpf$", t)
        if m:
            return t, int(m.group(1))
    # handle split forms like '72', 'hpf'
    for i, t in enumerate(tokens[:-1]):
        if re.fullmatch(r"[0-9]+", t) and tokens[i+1] == "hpf":
            return f"{t}hpf", int(t)
    return None, None

def infer_roi_index(tokens):
    """
    Look for ROI index tokens like 'roi1', 'roi2'.
    Returns (roi_token, roi_index:int) or (None, None).
    """
    for t in tokens:
        m = re.match(r"^roi\s*0*([0-9]+)$", t)
        if m:
            return t, int(m.group(1))
    return None, None

def infer_fish_id_from_number_or_fish_col(row, fish_number):
    """
    Decide fish_id.
      - If fish_number is not None, use 'fish<fish_number>'.
      - Else, fall back to 'fish' column if present.
      - Else, last resort: use fish_folder as-is.
    """
    if fish_number is not None:
        return f"fish{fish_number}"

    # try 'fish' column
    if "fish" in row and pd.notna(row["fish"]):
        s = str(row["fish"]).strip().lower()
        m = re.search(r"fish\s*0*([0-9]+)", s)
        if m:
            return f"fish{int(m.group(1))}"
        # bare number
        m = re.fullmatch(r"0*([0-9]+)", s)
        if m:
            return f"fish{int(m.group(1))}"
        return s or None

    # fallback: fish_folder
    ff = row.get("fish_folder")
    return ff if ff is not None else None

def infer_fish_nickname(fish_folder, fish_number, age_token, roi_token):
    """
    Very rough attempt to get a "nickname" piece from fish_folder.

    Strategy:
      - Take fish_folder string.
      - Remove 'fish<number>', age_token, and roi_token substrings.
      - Strip leftover underscores and spaces.
      - Whatever remains is fish_nickname (may be empty or None).
    """
    if fish_folder is None:
        return None
    s = str(fish_folder)

    if fish_number is not None:
        s = re.sub(rf"fish_?0?{fish_number}", "", s, flags=re.IGNORECASE)
        s = re.sub(rf"fish0?{fish_number}", "", s, flags=re.IGNORECASE)

    if age_token:
        s = s.replace(age_token, "")
        # also remove 'hpf' if it survived
        s = s.replace("hpf", "")

    if roi_token:
        s = s.replace(roi_token, "")

    # clean out stray underscores and whitespace
    s = re.sub(r"_+", " ", s)
    s = s.strip()
    return s or None

# --- apply inference per row --------------------------------------------------

fish_numbers = []
fish_ids     = []
fish_age_tokens = []
fish_age_hpfs   = []
roi_tokens      = []
roi_indices     = []
fish_nicknames  = []

for _, row in df_roi_paths.iterrows():
    path = row["roi_dir"]
    tokens = tokenize_path(path)

    fish_num = infer_fish_number(tokens)
    age_tok, age_hpf = infer_fish_age(tokens)
    roi_tok, roi_idx = infer_roi_index(tokens)
    fish_id = infer_fish_id_from_number_or_fish_col(row, fish_num)
    nickname = infer_fish_nickname(row.get("fish_folder"), fish_num, age_tok, roi_tok)

    fish_numbers.append(fish_num)
    fish_ids.append(fish_id)
    fish_age_tokens.append(age_tok)
    fish_age_hpfs.append(age_hpf)
    roi_tokens.append(roi_tok)
    roi_indices.append(roi_idx)
    fish_nicknames.append(nickname)

df_roi_paths["fish_number"]     = fish_numbers
df_roi_paths["fish_id"]         = fish_ids
df_roi_paths["fish_age_token"]  = fish_age_tokens
df_roi_paths["fish_age_hpf"]    = fish_age_hpfs
df_roi_paths["roi_index_token"] = roi_tokens
df_roi_paths["roi_index"]       = roi_indices
df_roi_paths["fish_nickname"]   = fish_nicknames

# --- QC: show some mappings so you can spot-check -----------------------------

print("L2c: sample parsed fields from roi_dir (spot-check these):")
print(
    df_roi_paths[
        [
            "roi_dir",
            "experiment_folder",
            "fish_folder",
            "fish_number",
            "fish_id",
            "fish_age_token",
            "fish_age_hpf",
            "roi_index_token",
            "roi_index",
            "fish_nickname",
        ]
    ].head(40)
)

print("\nL2c: distinct fish_folder -> (fish_number, fish_id, fish_age_token, fish_nickname) (first 40):")
print(
    df_roi_paths[
        ["experiment_folder", "fish_folder", "fish_number", "fish_id",
         "fish_age_token", "fish_nickname"]
    ]
    .drop_duplicates()
    .head(40)
)

L2c: sample parsed fields from roi_dir (spot-check these):
                                              roi_dir     experiment_folder  \
0   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250428_mem_histone   
1   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250428_mem_histone   
2   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
3   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
4   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
5   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
6   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
7   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
8   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
9   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
10  /clusterfs/vast/abcabc/Korra_Foundation/202505...     20250513_skitt

In [169]:
# LINKING_V4 — L2d: extract roi_anatomy from roi_folder using tokenization

import re
import pandas as pd

df_roi_paths = df_roi_paths.copy()

def extract_roi_anatomy_tokens(roi_folder: str):
    """
    Parse roi_folder into anatomy tokens.

    Heuristic:
      1. Split roi_folder on '_' to tokens.
      2. Find first 'roiNN' or 'roi'.
      3. Take tokens after that as candidates.
      4. Drop age-like tokens: 'NNhpf', 'NN', 'hpf', etc.
      5. Return list of remaining tokens (possibly empty).
    """
    if roi_folder is None:
        return []
    s = str(roi_folder).strip().lower()
    tokens = [t for t in s.split("_") if t]

    # find roi index position
    roi_pos = None
    for i, t in enumerate(tokens):
        if re.match(r"^roi[0-9]*$", t):  # roi, roi1, roi2, etc.
            roi_pos = i
            break

    if roi_pos is None:
        # no explicit roi token; treat all tokens as candidates but still filter age
        cand = tokens
    else:
        cand = tokens[roi_pos+1:]

    anatomy = []
    for t in cand:
        # age-like tokens: 24hpf, 72hpf, 48hpf, 24, 72, 'hpf'
        if re.match(r"^[0-9]+hpf$", t):
            continue
        if re.fullmatch(r"[0-9]+", t):
            continue
        if t == "hpf":
            continue
        anatomy.append(t)

    return anatomy

# apply
df_roi_paths["roi_anatomy_tokens"] = df_roi_paths["roi_folder"].apply(extract_roi_anatomy_tokens)
df_roi_paths["roi_anatomy"] = df_roi_paths["roi_anatomy_tokens"].apply(
    lambda toks: "|".join(toks) if toks else None
)

print("L2d: sample roi_folder -> roi_anatomy_tokens / roi_anatomy:")
print(
    df_roi_paths[
        ["roi_dir", "roi_folder", "roi_anatomy_tokens", "roi_anatomy"]
    ].head(40)
)

print("\nL2d: distinct roi_folder -> roi_anatomy (first 40):")
print(
    df_roi_paths[
        ["experiment_folder", "roi_folder", "roi_anatomy"]
    ]
    .drop_duplicates()
    .head(40)
)

L2d: sample roi_folder -> roi_anatomy_tokens / roi_anatomy:
                                              roi_dir            roi_folder  \
0   /clusterfs/vast/abcabc/Korra_Foundation/202504...             roi1_tail   
1   /clusterfs/vast/abcabc/Korra_Foundation/202504...  roi2_hindbrain_spine   
2   /clusterfs/vast/abcabc/Korra_Foundation/202504...                  roi1   
3   /clusterfs/vast/abcabc/Korra_Foundation/202504...             roi1_test   
4   /clusterfs/vast/abcabc/Korra_Foundation/202504...                  roi1   
5   /clusterfs/vast/abcabc/Korra_Foundation/202504...                  roi1   
6   /clusterfs/vast/abcabc/Korra_Foundation/202504...                  roi2   
7   /clusterfs/vast/abcabc/Korra_Foundation/202504...                  roi1   
8   /clusterfs/vast/abcabc/Korra_Foundation/202504...                  roi2   
9   /clusterfs/vast/abcabc/Korra_Foundation/202504...                  roi3   
10  /clusterfs/vast/abcabc/Korra_Foundation/202505...                  

In [170]:
# LINKING_V4 — L3: prepare imaging sheet (imaging_date_from_sheet + dataset_slug)

import pandas as pd
import re
import math

if "Date imaged" not in df_imaging.columns:
    raise KeyError("'Date imaged' column not found in df_imaging.")
if "Data location" not in df_imaging.columns:
    raise KeyError("'Data location' column not found in df_imaging.")

def extract_dataset_slugs_from_dataloc(path):
    """
    From Data location like:
      X:\\abcabc\\Korra_Foundation\\20250627_skittlez and 20250625_skittlez

    extract ALL path-like tokens that look like 'YYYYMMDD_<something>'.

    Returns:
      list of candidates (possibly empty).
    """
    if path is None or (isinstance(path, float) and pd.isna(path)):
        return []
    s = str(path).replace("\\", "/")
    # split on slashes and whitespace and commas and 'and'
    tokens = re.split(r"[\/\s,]+", s)
    cands = []
    for t in tokens:
        m = re.match(r"(\d{8}_.+)", t)
        if m:
            # strip trailing '(...)' if present
            slug = re.sub(r"\(.*\)$", "", m.group(1)).strip()
            cands.append(slug)
    return cands

df_imaging = df_imaging.copy()

# imaging_date_from_sheet = Date imaged as YYYYMMDD
df_imaging["imaging_date_from_sheet"] = pd.to_datetime(
    df_imaging["Date imaged"], errors="coerce"
).dt.strftime("%Y%m%d")

# ALL slug candidates per row
df_imaging["slug_candidates"] = df_imaging["Data location"].apply(extract_dataset_slugs_from_dataloc)

def pick_dataset_slug(row):
    """
    Choose a single dataset_slug from slug_candidates using imaging_date_from_sheet:

      - If no candidates: return None.
      - If exactly one candidate: return it.
      - If multiple candidates:
           Try to keep those whose 8-digit prefix matches imaging_date_from_sheet.
           If exactly one such candidate: return it.
           Else: return None and let QC handle it.
    """
    cands = row["slug_candidates"]
    if not cands:
        return None

    if len(cands) == 1:
        return cands[0]

    img_date = row["imaging_date_from_sheet"]
    if img_date is None or isinstance(img_date, float) and math.isnan(img_date):
        # can't resolve tie without a date; treat as ambiguous
        return None

    # filter candidates by date prefix
    matched = [c for c in cands if c.startswith(img_date + "_")]
    if len(matched) == 1:
        return matched[0]

    # ambiguous or still multiple; return None to force manual fix
    return None

df_imaging["dataset_slug"] = df_imaging.apply(pick_dataset_slug, axis=1)

print("L3: sample imaging rows with imaging_date_from_sheet and dataset_slug:")
print(
    df_imaging[
        ["Date imaged", "imaging_date_from_sheet", "Data location", "slug_candidates", "dataset_slug"]
    ].head(20)
)

L3: sample imaging rows with imaging_date_from_sheet and dataset_slug:
   Date imaged imaging_date_from_sheet  \
0          NaT                     NaN   
1   2025-11-21                20251121   
2   2025-11-17                20251117   
3   2025-11-14                20251114   
4   2025-11-14                20251114   
5   2025-11-13                20251113   
6   2025-11-12                20251112   
7   2025-11-12                20251112   
8   2025-11-11                20251111   
9   2025-11-10                20251110   
10  2025-11-06                20251106   
11  2025-11-05                20251105   
12  2025-11-05                20251105   
13  2025-10-29                20251029   
14  2025-10-30                20251030   
15  2025-10-28                20251028   
16  2025-10-28                20251028   
17  2025-10-29                20251029   
18  2025-10-27                20251027   
19  2025-10-27                20251027   

                                        Data l

In [171]:
# LINKING_V4 — L4: attach dataset_slug + QC on ROI side

# On ROI side, dataset_slug is just experiment_folder (e.g. '20250513_skittlez')
df_roi_paths["dataset_slug"] = df_roi_paths["experiment_folder"]

print("L4: sample ROI rows with dataset_slug + roi_path_date_yyyymmdd:")
print(
    df_roi_paths[
        ["roi_dir", "experiment_folder", "dataset_slug", "roi_path_date_yyyymmdd"]
    ].head(20)
)

L4: sample ROI rows with dataset_slug + roi_path_date_yyyymmdd:
                                              roi_dir     experiment_folder  \
0   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250428_mem_histone   
1   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250428_mem_histone   
2   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
3   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
4   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
5   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
6   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
7   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
8   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
9   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
10  /clusterfs/vast/abcabc/Korra_Foundation/202505...     20250513_

In [172]:
# L5 — link ROI rows to imaging-sheet rows by normalized slug + mount date

# Build a compact imaging key table with the fields we care about for linking
imaging_key_cols = [
    "sheet_slug_norm",
    "date_mount_yyyymmdd",
    "Data location",
    # genotype / treatment columns we want later:
    "ZF female genotype",
    "ZF male genotype",
    "additional plasmids injected",
    "additional mRNAs injected",
    "additonal proteins injected",
    "additonal dye and chemicals",
]

missing_imaging_cols = [c for c in imaging_key_cols if c not in df_imaging.columns]
if missing_imaging_cols:
    raise KeyError(f"Imaging sheet is missing expected columns: {missing_imaging_cols}")

df_imaging_keys = df_imaging[imaging_key_cols].copy()

print("L5 — df_imaging_keys sample:")
print(df_imaging_keys.head(5))

# Merge by:
#   ROI side:   (dataset_slug_norm, roi_path_date_yyyymmdd)
#   Imaging:    (sheet_slug_norm,  date_mount_yyyymmdd)
df_linked = df_roi_paths.merge(
    df_imaging_keys,
    left_on=["dataset_slug_norm", "roi_path_date_yyyymmdd"],
    right_on=["sheet_slug_norm", "date_mount_yyyymmdd"],
    how="left",
    suffixes=("", "_img"),
)

print("\nL5 — df_linked rows:", len(df_linked), "unique roi_dir:", df_linked["roi_dir"].nunique())

# Mark link_source based on whether an imaging row was found
df_linked["link_source"] = df_linked["Data location"].where(
    df_linked["Data location"].notna(), other="inferred"
)
df_linked["link_source"] = df_linked["link_source"].apply(
    lambda s: "sheet" if isinstance(s, str) and s != "inferred" else "inferred"
)

print("L5 — link_source breakdown:")
print(df_linked["link_source"].value_counts())

# Quick QC: show some rows that are still inferred (i.e., no imaging match even after slug normalization)
print("\nL5 — sample inferred rows (no imaging match):")
display(
    df_linked.loc[
        df_linked["link_source"].eq("inferred"),
        ["roi_dir", "dataset_slug_raw", "dataset_slug_norm", "roi_path_date_yyyymmdd"],
    ].head(20)
)

L5 — df_imaging_keys sample:
        sheet_slug_norm date_mount_yyyymmdd  \
0                  None            20251121   
1  20251121_mem_histone            20251121   
2  20251117_mem_histone            20251117   
3     20251113_mem_mito            20251114   
4     20251113_mem_mito            20251114   

                                       Data location  \
0                                                NaN   
1    X:\abcabc\Korra_Foundation\20251121_mem-histone   
2    X:\abcabc\Korra_Foundation\20251117_mem-histone   
3  X:\abcabc\Korra_Foundation\20251113_mem-mito_m...   
4  X:\abcabc\Korra_Foundation\20251113_mem-mito_m...   

                          ZF female genotype  \
0    ef1a:2xLynk:tdmSG(J) (F1 of allele 301)   
1    ef1a:2xLynk:tdmSG(J) (F1 of allele 301)   
2    ef1a:2xLynk:tdmSG(J) (F1 of allele 302)   
3  ef1a:2xLynk:tdmChilada (F1 of allele 315)   
4   memrane tdmScarlet3S2 (F1 of allele 325)   

                            ZF male genotype  \
0  tdmScarlet3

,roi_dir,dataset_slug_raw,dataset_slug_norm,roi_path_date_yyyymmdd
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,korra,korra,20250428
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,korra,korra,20250428
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,korra,korra,20250429
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,korra,korra,20250429
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,korra,korra,20250429
5,/clusterfs/vast/abcabc/Korra_Foundation/202504...,korra,korra,20250429
6,/clusterfs/vast/abcabc/Korra_Foundation/202504...,korra,korra,20250429
7,/clusterfs/vast/abcabc/Korra_Foundation/202504...,korra,korra,20250429
8,/clusterfs/vast/abcabc/Korra_Foundation/202504...,korra,korra,20250429
9,/clusterfs/vast/abcabc/Korra_Foundation/202504...,korra,korra,20250429


In [173]:
# L5b — date-only fallback linking for "hole" ROIs using exact mount date + bio-profile dedupe

import numpy as np
import pandas as pd
import re
from pathlib import Path

print("\nL5b — date-only fallback linking for hole ROIs (bio-profile dedupe)")

# -------------------------------------------------------------------
# 1) Load manual "holes" list (ROIs that need special attention)
# -------------------------------------------------------------------

HOLES_PATH = RAW / "roi_missing_parents_for_manual_mapping_DQM_CNH (1).csv"
print("  Manual holes file:", HOLES_PATH)

hole_roi_dirs: set[str] = set()
if HOLES_PATH.exists():
    manual_holes = pd.read_csv(HOLES_PATH)
    if "roi_dir" not in manual_holes.columns:
        raise KeyError(f"Manual holes file {HOLES_PATH} missing 'roi_dir' column.")
    hole_roi_dirs = set(manual_holes["roi_dir"].dropna().astype(str))
    print("  manual holes rows:", len(manual_holes), "unique roi_dir:", len(hole_roi_dirs))
else:
    print("  [WARN] Manual holes file not found; fallback will use link_source=='inferred' only.")
    manual_holes = None

df_linked = df_linked.copy()

# -------------------------------------------------------------------
# 2) Ensure ROI date + link_source exist on df_linked
# -------------------------------------------------------------------

if "roi_path_date_yyyymmdd" not in df_linked.columns:
    def roi_date_from_path(s):
        if pd.isna(s):
            return None
        s = str(s)
        m = re.search(r"/(\d{8})_", s)
        if m:
            return m.group(1)
        return None
    df_linked["roi_path_date_yyyymmdd"] = df_linked["roi_dir"].apply(roi_date_from_path)

if "link_source" not in df_linked.columns:
    df_linked["link_source"] = np.where(
        df_linked["Data location"].notna(), "sheet", "inferred"
    )

# -------------------------------------------------------------------
# 3) Identify hole rows that still lack an imaging-sheet match
# -------------------------------------------------------------------

mask_no_sheet = df_linked["Data location"].isna()

if hole_roi_dirs:
    mask_hole = mask_no_sheet & df_linked["roi_dir"].astype(str).isin(hole_roi_dirs)
else:
    mask_hole = mask_no_sheet

n_hole = mask_hole.sum()
print("  hole rows with no imaging match (before date-fallback):", n_hole)

if n_hole == 0:
    print("  Nothing to do; all hole rows already have imaging-sheet matches.")
else:
    # -------------------------------------------------------------------
    # 4) Ensure df_imaging has date_mount_yyyymmdd
    # -------------------------------------------------------------------
    if "date_mount_yyyymmdd" not in df_imaging.columns:
        date_mount_col = None
        for cand in ["date_mount", "Date mount", "Date mounted", "Date mounted?"]:
            if cand in df_imaging.columns:
                date_mount_col = cand
                break
        if date_mount_col is None:
            raise KeyError("df_imaging missing date_mount column for fallback linking.")

        df_imaging["date_mount_yyyymmdd"] = pd.to_datetime(
            df_imaging[date_mount_col], errors="coerce"
        ).dt.strftime("%Y%m%d")

    # Bio-identity keys for deduplication
    bio_keys = [
        "sheet_slug_norm",
        "ZF female genotype",
        "ZF male genotype",
        "additional plasmids injected",
        "additional mRNAs injected",
        "additonal proteins injected",
        "additonal dye and chemicals",
    ]
    missing_bio = [c for c in bio_keys if c not in df_imaging.columns]
    if missing_bio:
        raise KeyError(f"df_imaging missing expected bio-identity columns: {missing_bio}")

    # Dates present in hole rows
    hole_dates = set(
        df_linked.loc[mask_hole, "roi_path_date_yyyymmdd"].dropna().astype(str).unique()
    )
    print("  unique hole dates:", len(hole_dates))

    # -------------------------------------------------------------------
    # 5) For each hole date, decide if there is a unique bio profile
    # -------------------------------------------------------------------
    safe_date_to_row = {}

    for d in sorted(hole_dates):
        sub = df_imaging[df_imaging["date_mount_yyyymmdd"] == d]
        if sub.empty:
            continue

        bio_unique = sub.drop_duplicates(subset=bio_keys)

        n_unique = len(bio_unique)
        if n_unique == 1:
            safe_date_to_row[d] = bio_unique.iloc[0]
        # if >1 unique bio profiles, we treat as ambiguous and skip

    safe_dates = set(safe_date_to_row.keys())
    print("  safe hole dates (exactly 1 unique bio profile on that date):", len(safe_dates))

    # Build a small DataFrame of safe imaging rows
    if safe_dates:
        imaging_safe = pd.DataFrame.from_records(list(safe_date_to_row.values()))
    else:
        imaging_safe = pd.DataFrame(columns=df_imaging.columns)

    # We only need these imaging columns to carry over
    imaging_cols = [
        "date_mount_yyyymmdd",
        "Data location",
        "ZF female genotype",
        "ZF male genotype",
        "additional plasmids injected",
        "additional mRNAs injected",
        "additonal proteins injected",
        "additonal dye and chemicals",
    ]
    imaging_cols = [c for c in imaging_cols if c in imaging_safe.columns]
    imaging_safe = imaging_safe[imaging_cols].copy()

    # -------------------------------------------------------------------
    # 6) Merge safe-date imaging info onto hole rows by date only
    # -------------------------------------------------------------------
    hole_subset = df_linked.loc[mask_hole].copy()

    hole_subset = hole_subset.merge(
        imaging_safe,
        left_on="roi_path_date_yyyymmdd",
        right_on="date_mount_yyyymmdd",
        how="left",
        suffixes=("", "_date"),
    )

    # rows that actually got a date-based imaging match
    got_date_match = hole_subset["Data location_date"].notna() if "Data location_date" in hole_subset.columns else pd.Series(False, index=hole_subset.index)
    n_date_matched = got_date_match.sum()
    print("  hole rows with date-based imaging match:", n_date_matched)

    # combine-first for the imaging fields
    for col in [
        "Data location",
        "ZF female genotype",
        "ZF male genotype",
        "additional plasmids injected",
        "additional mRNAs injected",
        "additonal proteins injected",
        "additonal dye and chemicals",
    ]:
        date_col = f"{col}_date"
        if date_col in hole_subset.columns:
            hole_subset[col] = hole_subset[col].combine_first(hole_subset[date_col])
            hole_subset.drop(columns=[date_col], inplace=True)

    # drop helper join columns if present
    for helper_col in ["date_mount_yyyymmdd_date"]:
        if helper_col in hole_subset.columns:
            hole_subset.drop(columns=[helper_col], inplace=True)

    # update link_source only for rows that got a date match
    hole_subset.loc[got_date_match, "link_source"] = "date_match"

    # -------------------------------------------------------------------
    # 7) Write patched hole subset back into df_linked
    # -------------------------------------------------------------------
    common_cols = [c for c in df_linked.columns if c in hole_subset.columns]
    df_linked.loc[mask_hole, common_cols] = hole_subset[common_cols]

    # -------------------------------------------------------------------
    # 8) Report any remaining hole rows with no sheet/date match
    # -------------------------------------------------------------------
    still_hole = mask_hole & df_linked["Data location"].isna()
    n_still = still_hole.sum()
    print("  hole rows still without imaging match after date+bio fallback:", n_still)

    if n_still:
        out_path = WORKING / "linking_v4_hole_rows_needing_manual_review.csv"
        df_linked.loc[still_hole].to_csv(out_path, index=False)
        print("  Wrote remaining hole rows needing manual review to:", out_path)

print("\nL5b — link_source breakdown after date+bio fallback:")
print(df_linked["link_source"].value_counts())


L5b — date-only fallback linking for hole ROIs (bio-profile dedupe)
  Manual holes file: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/roi_missing_parents_for_manual_mapping_DQM_CNH (1).csv
  manual holes rows: 172 unique roi_dir: 172
  hole rows with no imaging match (before date-fallback): 171
  unique hole dates: 18
  safe hole dates (exactly 1 unique bio profile on that date): 9
  hole rows with date-based imaging match: 92
  hole rows still without imaging match after date+bio fallback: 111
  Wrote remaining hole rows needing manual review to: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/linking_v4_hole_rows_needing_manual_review.csv

L5b — link_source breakdown after date+bio fallback:
link_source
inferred      809
date_match     60
Name: count, dtype: int64


In [174]:
# LINKING_V4 — L6: infer mount_id from fish_id, then derive plate/slot/ROI indices

import pandas as pd
import numpy as np

df_linked = df_linked.copy()

# ─────────────────────────────────────────────────────────────
# 0) Drop rows with null roi_dir (non-real ROIs)
# ─────────────────────────────────────────────────────────────
null_roi_mask = df_linked["roi_dir"].isna()
n_null_roi = null_roi_mask.sum()
print(f"L6: rows with null roi_dir (will be dropped): {n_null_roi}")
if n_null_roi:
    bad_null_path = WORKING / "linking_v4_null_roi_dir_rows.csv"
    df_linked.loc[null_roi_mask].to_csv(bad_null_path, index=False)
    print("     wrote null-roi_dir rows to:", bad_null_path)
    df_linked = df_linked.loc[~null_roi_mask].copy()

# ─────────────────────────────────────────────────────────────
# 1) Choose a mount-date source and compute plate_date
# ─────────────────────────────────────────────────────────────
mount_date_col = None

for cand in ["date_mount", "Date mount", "Date mounted"]:
    if cand in df_linked.columns:
        mount_date_col = cand
        break

if mount_date_col is None:
    # fallback: use imaging date as mount date surrogate
    for cand in ["Date imaged", "date_imaged"]:
        if cand in df_linked.columns:
            mount_date_col = cand
            break

if mount_date_col is not None:
    print(f"L6: using mount_date_col={mount_date_col!r} for plate_date")
    mount_dt = pd.to_datetime(df_linked[mount_date_col], errors="coerce")
    bad_mask = mount_dt.isna()
    if bad_mask.any():
        bad = df_linked[bad_mask]
        print("L6: rows with missing/invalid mount_date in", mount_date_col)
        print(bad[["roi_dir", "experiment_folder", "dataset_slug", mount_date_col, "link_source"]].head(30))
        out_bad_mount = WORKING / "linking_v4_missing_mount_date_rows.csv"
        bad[["roi_dir", "experiment_folder", "dataset_slug",
             mount_date_col, "Data location", "link_source"]].to_csv(
            out_bad_mount, index=False
        )
        print("\nL6: wrote rows with missing/invalid mount_date to:", out_bad_mount)
        raise ValueError(f"L6: {mount_date_col} could not be parsed for some rows; fix upstream.")
    df_linked["plate_date"] = mount_dt.dt.strftime("%Y%m%d")
else:
    # last-resort: use roi_path_date_yyyymmdd (from ROI path)
    if "roi_path_date_yyyymmdd" not in df_linked.columns:
        raise KeyError(
            "L6: No mount/imaging date column found and roi_path_date_yyyymmdd missing. "
            "Cannot infer plate_date."
        )
    print("L6: using roi_path_date_yyyymmdd for plate_date (no explicit mount/imaging date column found)")
    if df_linked["roi_path_date_yyyymmdd"].isna().any():
        bad = df_linked[df_linked["roi_path_date_yyyymmdd"].isna()]
        print("L6: rows with missing roi_path_date_yyyymmdd (after dropping null roi_dir):")
        print(bad[["roi_dir", "experiment_folder", "dataset_slug", "link_source"]].head(30))
        out_bad_mount = WORKING / "linking_v4_missing_roi_path_date_rows.csv"
        bad[["roi_dir", "experiment_folder", "dataset_slug",
             "roi_path_date_yyyymmdd", "Data location", "link_source"]].to_csv(
            out_bad_mount, index=False
        )
        print("\nL6: wrote rows with missing roi_path_date_yyyymmdd to:", out_bad_mount)
        raise ValueError("L6: roi_path_date_yyyymmdd is NaN for some rows; fix ROI paths.")
    df_linked["plate_date"] = df_linked["roi_path_date_yyyymmdd"].astype(str)

# ─────────────────────────────────────────────────────────────
# 2) Ensure fish_id exists
# ─────────────────────────────────────────────────────────────
if "fish_id" not in df_linked.columns:
    raise KeyError("L6: df_linked missing 'fish_id' column; update L2c to compute it.")

# ─────────────────────────────────────────────────────────────
# 3) Infer mount_id_inferred per (plate_date, experiment_folder)
# ─────────────────────────────────────────────────────────────
def infer_mount_id_per_group(fish_ids: pd.Series) -> pd.Series:
    """
    Given fish_id values for a (plate_date, experiment_folder),
    assign mount IDs so that each mount has up to 6 unique fish_id entries.

    Steps:
      - Get sorted unique fish_id
      - Map each fish_id to an index 0..N-1
      - mount_id_inferred = idx // 6 + 1
    """
    uniques = sorted(fish_ids.unique())
    idx_map = {f: i for i, f in enumerate(uniques)}
    idx = fish_ids.map(idx_map)
    return (idx // 6) + 1

df_linked["mount_id_inferred"] = (
    df_linked
    .sort_values(["plate_date", "experiment_folder", "fish_id"])
    .groupby(["plate_date", "experiment_folder"])["fish_id"]
    .transform(infer_mount_id_per_group)
)

df_linked["mount_id_source"] = "inferred"

# ─────────────────────────────────────────────────────────────
# 4) plate_key and plate_id_filled
# ─────────────────────────────────────────────────────────────
df_linked["plate_key"] = (
    df_linked["plate_date"].astype(str)
    + "::" + df_linked["experiment_folder"].astype(str)
    + "::" + df_linked["mount_id_inferred"].astype(int).astype(str)
)

df_linked["plate_id_filled"] = pd.factorize(df_linked["plate_key"])[0] + 1

# ─────────────────────────────────────────────────────────────
# 5) slot_id_filled: per plate, index fish_id
# ─────────────────────────────────────────────────────────────
if "roi_folder" not in df_linked.columns:
    raise KeyError("L6: df_linked missing 'roi_folder' column; update L2c to compute it.")

df_linked["slot_id_filled"] = (
    df_linked
    .sort_values(["plate_id_filled", "fish_id"])
    .groupby("plate_id_filled")["fish_id"]
    .transform(lambda s: pd.factorize(s, sort=True)[0] + 1)
)

# ─────────────────────────────────────────────────────────────
# 6) roi_index_within_slot: per (plate_id, slot_id), index roi_folder
# ─────────────────────────────────────────────────────────────
df_linked["roi_index_within_slot"] = (
    df_linked
    .sort_values(["plate_id_filled", "slot_id_filled", "roi_folder"])
    .groupby(["plate_id_filled", "slot_id_filled"])["roi_folder"]
    .cumcount() + 1
)

# ─────────────────────────────────────────────────────────────
# 7) QC: unique fish_id per inferred plate must be <= 6
# ─────────────────────────────────────────────────────────────
fish_counts = (
    df_linked
    .groupby(["plate_date", "experiment_folder", "mount_id_inferred", "plate_id_filled"])["fish_id"]
    .nunique()
    .reset_index(name="n_fish")
)

bad_plates = fish_counts[fish_counts["n_fish"] > 6]
print("L6: inferred plates with >6 fish_id (should be 0):", len(bad_plates))
if len(bad_plates):
    print(bad_plates.head(20))
    out_bad_plates = WORKING / "linking_v4_bad_plates_n_fish_gt6_INFERRED.csv"
    bad_plates.to_csv(out_bad_plates, index=False)
    print("\nL6: wrote bad inferred plates summary to:", out_bad_plates)
    raise ValueError("L6: found inferred plates with more than 6 fish; check inference logic.")

print("\nL6: sample plate/slot/ROI hierarchy (inferred mount IDs):")
cols_to_show = [
    "roi_dir",
    "plate_date",
    "experiment_folder",
    "mount_id_inferred",
    "plate_id_filled",
    "fish_id",
    "slot_id_filled",
    "roi_folder",
    "roi_index_within_slot",
]
if "Date imaged" in df_linked.columns:
    cols_to_show.append("Date imaged")
cols_to_show.append("link_source")

print(df_linked[cols_to_show].head(30))

L6: rows with null roi_dir (will be dropped): 106
     wrote null-roi_dir rows to: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/linking_v4_null_roi_dir_rows.csv
L6: using roi_path_date_yyyymmdd for plate_date (no explicit mount/imaging date column found)
L6: inferred plates with >6 fish_id (should be 0): 0

L6: sample plate/slot/ROI hierarchy (inferred mount IDs):
                                              roi_dir plate_date  \
0   /clusterfs/vast/abcabc/Korra_Foundation/202504...   20250428   
1   /clusterfs/vast/abcabc/Korra_Foundation/202504...   20250428   
2   /clusterfs/vast/abcabc/Korra_Foundation/202504...   20250429   
3   /clusterfs/vast/abcabc/Korra_Foundation/202504...   20250429   
4   /clusterfs/vast/abcabc/Korra_Foundation/202504...   20250429   
5   /clusterfs/vast/abcabc/Korra_Foundation/202504...   20250429   
6   /clusterfs/vast/abcabc/Korra_Foundation/202504...   20250429   
7   /clusterfs/vast/abcabc/Korra_Foundation/202504...   202504

In [175]:
# QC — 20250606 fish_id per inferred mount
mask_0606 = df_linked["plate_date"] == "20250606"
fish_counts_0606 = (
    df_linked[mask_0606]
    .groupby(["experiment_folder", "mount_id_inferred"])["fish_id"]
    .nunique()
    .reset_index(name="n_fish")
    .sort_values(["experiment_folder", "mount_id_inferred"])
)
print(fish_counts_0606)

   experiment_folder  mount_id_inferred  n_fish
0  20250606_skittlez                  1       6
1  20250606_skittlez                  2       6
2  20250606_skittlez                  3       2


In [176]:
# LINKING_V4 — L7: construct bruker_roi_id

def make_bruker_roi_id(row):
    # bruker ID = YYYYMMDD-plateN-slotN-roiN
    date = row["plate_date"]
    plate = row["plate_id_filled"]
    slot  = row["slot_id_filled"]
    idx   = row["roi_index_within_slot"]
    return f"{date}-plate{plate}-slot{slot}-roi{idx}"

df_linked = df_linked.copy()
df_linked["bruker_roi_id"] = df_linked.apply(make_bruker_roi_id, axis=1)

print("L7: sample bruker_roi_id values:")
print(
    df_linked[
        ["roi_dir", "bruker_roi_id", "plate_date",
         "plate_id_filled", "slot_id_filled", "roi_index_within_slot"]
    ].head(15)
)

L7: sample bruker_roi_id values:
                                              roi_dir  \
0   /clusterfs/vast/abcabc/Korra_Foundation/202504...   
1   /clusterfs/vast/abcabc/Korra_Foundation/202504...   
2   /clusterfs/vast/abcabc/Korra_Foundation/202504...   
3   /clusterfs/vast/abcabc/Korra_Foundation/202504...   
4   /clusterfs/vast/abcabc/Korra_Foundation/202504...   
5   /clusterfs/vast/abcabc/Korra_Foundation/202504...   
6   /clusterfs/vast/abcabc/Korra_Foundation/202504...   
7   /clusterfs/vast/abcabc/Korra_Foundation/202504...   
8   /clusterfs/vast/abcabc/Korra_Foundation/202504...   
9   /clusterfs/vast/abcabc/Korra_Foundation/202504...   
10  /clusterfs/vast/abcabc/Korra_Foundation/202505...   
11  /clusterfs/vast/abcabc/Korra_Foundation/202505...   
12  /clusterfs/vast/abcabc/Korra_Foundation/202505...   
13  /clusterfs/vast/abcabc/Korra_Foundation/202505...   
14  /clusterfs/vast/abcabc/Korra_Foundation/202505...   

                 bruker_roi_id plate_date  plate_id_fi

In [177]:
# LINKING_V4 — L8: write final structural CSV

out_path = WORKING / "output_from_linking.csv"
df_linked.to_csv(out_path, index=False)

print("L8: wrote linking output to:", out_path)
print("L8: df_linked shape:", df_linked.shape)
print("L8: unique roi_dir:", df_linked["roi_dir"].nunique())
print("L8: link_source breakdown:")
print(df_linked["link_source"].value_counts())

L8: wrote linking output to: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/output_from_linking.csv
L8: df_linked shape: (869, 41)
L8: unique roi_dir: 869
L8: link_source breakdown:
link_source
inferred      809
date_match     60
Name: count, dtype: int64


In [179]:
# QC-A_v2 — Compare experiment_folder-derived date vs mount date used for linking

import re
import pandas as pd

df_qc_date = df_linked.copy()

def extract_date_prefix(exp_folder):
    if exp_folder is None or pd.isna(exp_folder):
        return None
    m = re.match(r"^(\d{8})_", str(exp_folder))
    return m.group(1) if m else None

# 1) experiment_folder-derived date (YYYYMMDD from leading 8 digits)
df_qc_date["date_mount_derived"] = df_qc_date["experiment_folder"].apply(extract_date_prefix)

# 2) choose the sheet-side date column (same priority idea as L6)
mount_date_col = None
for cand in ["date_mount", "Date mount", "Date mounted", "Date imaged", "date_imaged"]:
    if cand in df_qc_date.columns:
        mount_date_col = cand
        break

if mount_date_col is not None:
    print(f"QC-A_v2: using {mount_date_col!r} as sheet-side date for comparison")
    mount_dt = pd.to_datetime(df_qc_date[mount_date_col], errors="coerce")
    df_qc_date["date_mount_sheet"] = mount_dt.dt.strftime("%Y%m%d")
else:
    # fall back to plate_date if no explicit date column is present
    if "plate_date" not in df_qc_date.columns:
        raise KeyError(
            "QC-A_v2: no date_mount/Date imaged/plate_date column on df_linked; "
            "cannot run date comparison."
        )
    print("QC-A_v2: no explicit mount/imaging date; using plate_date for comparison")
    df_qc_date["date_mount_sheet"] = df_qc_date["plate_date"].astype(str)

# 3) basic QC stats
df_qc_valid = df_qc_date[~df_qc_date["date_mount_sheet"].isna() & ~df_qc_date["date_mount_derived"].isna()].copy()

n_total = len(df_qc_date)
n_valid = len(df_qc_valid)
n_match = (df_qc_valid["date_mount_derived"] == df_qc_valid["date_mount_sheet"]).sum()
n_mismatch = n_valid - n_match
n_missing_sheet = df_qc_date["date_mount_sheet"].isna().sum()
n_missing_derived = df_qc_date["date_mount_derived"].isna().sum()

print("QC-A_v2: total rows:", n_total)
print("QC-A_v2: rows with both dates present:", n_valid)
print("QC-A_v2: matches:", n_match)
print("QC-A_v2: mismatches:", n_mismatch)
print("QC-A_v2: missing sheet-side date:", n_missing_sheet)
print("QC-A_v2: missing experiment_folder-derived date:", n_missing_derived)

if n_mismatch > 0:
    print("\nQC-A_v2 — sample mismatches (up to 20):")
    print(
        df_qc_valid.loc[
            df_qc_valid["date_mount_derived"] != df_qc_valid["date_mount_sheet"],
            ["roi_dir", "experiment_folder", "date_mount_derived", "date_mount_sheet"]
        ].head(20)
    )

if n_missing_sheet > 0:
    print("\nQC-A_v2 — sample rows with missing sheet-side date (up to 20):")
    print(
        df_qc_date.loc[
            df_qc_date["date_mount_sheet"].isna(),
            ["roi_dir", "experiment_folder"] + ([mount_date_col] if mount_date_col else []) 
        ].head(20)
    )

QC-A_v2: no explicit mount/imaging date; using plate_date for comparison
QC-A_v2: total rows: 869
QC-A_v2: rows with both dates present: 869
QC-A_v2: matches: 869
QC-A_v2: mismatches: 0
QC-A_v2: missing sheet-side date: 0
QC-A_v2: missing experiment_folder-derived date: 0


In [180]:
# QC-B — Per imaging_date (YYYYMMDD) summary: n_plates, n_fish, n_rois, n_tiffs
# Updated: now uses marker='o' in all plots for clarity.

import pandas as pd
import matplotlib.pyplot as plt

df_qc = df_linked.copy()
df_qc["imaging_date"] = pd.to_datetime(df_qc["Date imaged"]).dt.strftime("%Y%m%d")

summary = (
    df_qc.groupby("imaging_date")
    .agg(
        n_plates = ("plate_id_filled", "nunique"),
        n_fish   = ("fish_id", "nunique"),
        n_rois   = ("roi_dir", "nunique"),
        n_tiffs  = ("roi_tiffs", "sum"),
    )
    .reset_index()
    .sort_values("imaging_date")
)

print("QC-B summary head:")
print(summary.head(20))

plt.figure(figsize=(28, 12))

plt.subplot(4,1,1)
plt.plot(summary["imaging_date"], summary["n_plates"], marker='o')
plt.title("Plates per Imaging Date")
plt.xticks(rotation=90)

plt.subplot(4,1,2)
plt.plot(summary["imaging_date"], summary["n_fish"], marker='o')
plt.title("Fish per Imaging Date")
plt.xticks(rotation=90)

plt.subplot(4,1,3)
plt.plot(summary["imaging_date"], summary["n_rois"], marker='o')
plt.title("ROIs per Imaging Date")
plt.xticks(rotation=90)

plt.subplot(4,1,4)
plt.plot(summary["imaging_date"], summary["n_tiffs"], marker='o')
plt.title("TIFF count per Imaging Date")
plt.xticks(rotation=90)

plt.tight_layout()
plt.show()

KeyError: 'Date imaged'

In [153]:
# QC-FISH — compare inferred fish_id vs raw 'fish' column and fish_folder

import pandas as pd

df_fish_qc = df_roi_paths.copy()

has_fish_col = "fish" in df_fish_qc.columns

print("QC-FISH: basic counts")
print("  total ROI rows:", len(df_fish_qc))
print("  unique fish_id:", df_fish_qc["fish_id"].nunique())
print("  unique fish_folder:", df_fish_qc["fish_folder"].nunique())
if has_fish_col:
    print("  unique raw 'fish' values:", df_fish_qc["fish"].nunique())
else:
    print("  NOTE: no 'fish' column in df_roi_paths; cannot compare to raw fish labels")

# Compare fish_id to normalized fish column where available
if has_fish_col:
    def norm_raw_fish(val):
        if pd.isna(val):
            return None
        s = str(val).strip().lower()
        m = re.search(r"fish\s*0*([0-9]+)", s)
        if m:
            return f"fish{int(m.group(1))}"
        m = re.fullmatch(r"0*([0-9]+)", s)
        if m:
            return f"fish{int(m.group(1))}"
        return s or None

    df_fish_qc["fish_raw_norm"] = df_fish_qc["fish"].apply(norm_raw_fish)
    df_fish_qc["fish_match"] = df_fish_qc["fish_id"] == df_fish_qc["fish_raw_norm"]

    n_with_raw = df_fish_qc["fish_raw_norm"].notna().sum()
    n_match = df_fish_qc["fish_match"].sum()
    n_mismatch = n_with_raw - n_match

    print("\nQC-FISH: comparison vs raw 'fish' column")
    print("  rows with non-null raw fish:", n_with_raw)
    print("  fish_id matches raw:", n_match)
    print("  fish_id mismatches:", n_mismatch)

    if n_mismatch:
        print("\nQC-FISH: sample mismatches (fish_folder, fish_raw_norm, fish_id):")
        print(
            df_fish_qc.loc[~df_fish_qc["fish_match"],
                           ["roi_dir", "fish_folder", "fish", "fish_raw_norm", "fish_id"]]
            .head(20)
        )

# Show distinct mapping from fish_folder to inferred fish_id
print("\nQC-FISH: distinct fish_folder -> fish_id (first 40):")
print(
    df_fish_qc[["experiment_folder", "fish_folder", "fish_number", "fish_id",
                "fish_age_token", "fish_nickname"]]
      .drop_duplicates()
      .head(40)
)

QC-FISH: basic counts
  total ROI rows: 975
  unique fish_id: 27
  unique fish_folder: 274
  unique raw 'fish' values: 275

QC-FISH: comparison vs raw 'fish' column
  rows with non-null raw fish: 975
  fish_id matches raw: 967
  fish_id mismatches: 8

QC-FISH: sample mismatches (fish_folder, fish_raw_norm, fish_id):
                                               roi_dir fish_folder  \
967  /clusterfs/vast/abcabc/Aang_Foundation/Denoisi...  fish1_roi1   
968  /clusterfs/vast/abcabc/Aang_Foundation/Denoisi...  fish3_roi1   
969  /clusterfs/vast/abcabc/Aang_Foundation/Denoisi...  fish4_roi1   
970  /clusterfs/vast/abcabc/Aang_Foundation/Denoisi...  fish5_roi1   
971  /clusterfs/vast/abcabc/Aang_Foundation/Denoisi...  fish5_roi2   
972  /clusterfs/vast/abcabc/Aang_Foundation/Denoisi...  fish5_roi3   
973  /clusterfs/vast/abcabc/Aang_Foundation/Denoisi...  fish1_roi1   
974  /clusterfs/vast/abcabc/Aang_Foundation/Denoisi...  fish3_roi1   

                     fish         fish_raw_norm fis

In [154]:
print(sheet.columns.tolist())

['date_mount', 'mount_id', 'ZF female genotype', 'ZF male genotype', 'additional plasmids injected', 'additional mRNAs injected', 'additonal proteins injected', 'additonal dye and chemicals', 'Date born', 'Time mounted', 'Mounting Orientation', 'Date screened/Initial feedback', 'Date imaged', 'Time placed in scope', 'Start of imaging time', 'End of imaging time', 'Imaged Locations', 'Unique Targets with blanks', 'Unique Targets', 'Data location', 'Dataset size (GB) - raw data only', 'Camera Filters', 'JSON excite map for ZF male', 'JSON excite map for ZF female', 'JSON excite map for plasmid', 'JSON excite map for mRNA', 'comments', 'Data evaluation comments', 'sheet_slug', 'date_mount_yyyymmdd', 'sheet_slug_norm']


In [155]:
from pathlib import Path
import pandas as pd

ROOT = Path("/Users/davekokel/Projects/carp_v2")
WORKING = ROOT / "seed_kits" / "legacy_wrangling_v2" / "working"

holes_path = WORKING / "linking_v4_hole_rows_needing_manual_review.csv"
holes = pd.read_csv(holes_path)

print("Remaining hole rows:", len(holes))
display(
    holes[
        [
            "roi_dir",
            "dataset_slug",
            "experiment_folder",
            "roi_path_date_yyyymmdd",
            "link_source",
        ]
    ]
)

Remaining hole rows: 111


,roi_dir,dataset_slug,experiment_folder,roi_path_date_yyyymmdd,link_source
0,/clusterfs/vast/abcabc/Korra_Foundation/202505...,20250528_skittlez,20250528_skittlez,20250528.0,inferred
1,/clusterfs/vast/abcabc/Korra_Foundation/202505...,20250528_skittlez,20250528_skittlez,20250528.0,inferred
2,/clusterfs/vast/abcabc/Korra_Foundation/202506...,20250620_mitomSG,20250620_mitomSG,20250620.0,inferred
3,/clusterfs/vast/abcabc/Korra_Foundation/202507...,20250724_mem-mchilada_er-mSG,20250724_mem-mchilada_er-mSG,20250724.0,inferred
4,/clusterfs/vast/abcabc/Korra_Foundation/202507...,20250724_mem-mchilada_er-mSG,20250724_mem-mchilada_er-mSG,20250724.0,inferred
...,...,...,...,...,...
106,NaN,NaN,NaN,NaN,NaN
107,NaN,NaN,NaN,NaN,NaN
108,NaN,NaN,NaN,NaN,NaN
109,NaN,NaN,NaN,NaN,NaN


In [156]:
import pandas as pd
from pathlib import Path

ROOT = Path("/Users/davekokel/Projects/carp_v2")
WORKING = ROOT / "seed_kits" / "legacy_wrangling_v2" / "working"

df_db = pd.read_csv(WORKING / "legacy_imaging_annotations_for_db_v4.csv")

print("rows:", len(df_db), "unique roi_dir:", df_db["roi_dir"].nunique())
print("\nlink_source breakdown:")
print(df_db["link_source"].value_counts())

print("\norganelles coverage:")
print(df_db["all_unique_organelles"].notna().value_counts())

print("\nexample rows with both genotype + treatment markers:")
cols = [
    "roi_dir",
    "dataset_slug",
    "experiment_folder",
    "fish_id",
    "genotype_base_codes",
    "treatment_plasmid_names_sheet",
    "treatment_rna_names_sheet",
    "genotype_marker_fusion_labels",
    "all_unique_organelles",
    "all_fluor_organelles",
]
display(df_db[df_db["genotype_base_codes"].notna()].head(20)[cols])

rows: 1118 unique roi_dir: 975

link_source breakdown:
link_source
sheet       990
inferred    128
Name: count, dtype: int64

organelles coverage:
all_unique_organelles
True     681
False    437
Name: count, dtype: int64

example rows with both genotype + treatment markers:


,roi_dir,dataset_slug,experiment_folder,fish_id,genotype_base_codes,treatment_plasmid_names_sheet,treatment_rna_names_sheet,genotype_marker_fusion_labels,all_unique_organelles,all_fluor_organelles
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250428_mem_histone,20250428_mem_histone,fish1,pDQM005,NaN,mScarlet3-S2:H2B,tdmSG::2xLynk,membrane,tdmSG(membrane)|mScarlet3S2(histone)
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250428_mem_histone,20250428_mem_histone,fish1,pDQM005,NaN,mScarlet3-S2:H2B,tdmSG::2xLynk,membrane,tdmSG(membrane)|mScarlet3S2(histone)
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,20250429_mem_cytosol,fish1,pDQM005,ef1a:mGold2s,NaN,tdmSG::2xLynk,membrane,tdmSG(membrane)
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,20250429_mem_cytosol,fish1,pDQM005,ef1a:mGold2s,NaN,tdmSG::2xLynk,membrane,tdmSG(membrane)
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,20250429_mem_cytosol,fish2,pDQM005,ef1a:mGold2s,NaN,tdmSG::2xLynk,membrane,tdmSG(membrane)
5,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,20250429_mem_cytosol,fish3,pDQM005,ef1a:mGold2s,NaN,tdmSG::2xLynk,membrane,tdmSG(membrane)
6,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,20250429_mem_cytosol,fish3,pDQM005,ef1a:mGold2s,NaN,tdmSG::2xLynk,membrane,tdmSG(membrane)
7,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,20250429_mem_cytosol,fish4,pDQM005,ef1a:mGold2s,NaN,tdmSG::2xLynk,membrane,tdmSG(membrane)
8,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,20250429_mem_cytosol,fish4,pDQM005,ef1a:mGold2s,NaN,tdmSG::2xLynk,membrane,tdmSG(membrane)
9,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,20250429_mem_cytosol,fish4,pDQM005,ef1a:mGold2s,NaN,tdmSG::2xLynk,membrane,tdmSG(membrane)
